# Week 6: Grounded answers with citations, refusal and answer language (Colab version)

Self-contained: choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**.
It writes the project code, collects the 38 official pages live, builds the 150 answerable and 50 unanswerable questions, calibrates the refusal threshold on all 200 and generates answers for a 10-topic subset (`--quick`, 50 questions in both modes). Remove `--quick` to run all 200 (much longer).

It downloads four models (about 15 GB in total; Qwen3.5-4B alone is about 9 GB). Pages are fetched live, so numbers can differ slightly from the 11 Nov 2026 run. Text in sections 5-7 describes the full local run.

In [ ]:
%pip install -q -U transformers sentence-transformers faiss-cpu rank-bm25 lingua-language-detector sentencepiece

In [ ]:
import os, sys
for d in ['src', 'eval', 'data/raw', 'data/processed', 'results']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile src/sources.py
"""Curated list of official sources for the three domains.

Every entry was checked against the live site on 28 Oct 2026 (HTTP 200).
GOV.UK pages are fetched through the public Content API; NHS pages as HTML.
Both are published under the Open Government Licence v3.0, and NHS reuse
also needs attribution, a link back and a date notice (nhs.uk terms).
"""

GOVUK = {
    "schools": [
        "schools-admissions",
        "apply-for-primary-school-place",
        "apply-for-secondary-school-place",
        "school-attendance-absence",
        "apply-free-school-meals",
        "children-with-special-educational-needs",
        "free-school-transport",
        "school-uniform",
        "types-of-school",
        "school-term-holiday-dates",
        "know-when-you-can-leave-school",
        "complain-about-school",
    ],
    "housing": [
        "private-renting",
        "private-renting-evictions",
        "guidance/renters-rights-act-overview-for-tenants",
        "tenancy-deposit-protection",
        "council-housing",
        "housing-association-homes",
        "council-housing-association-evictions",
        "housing-benefit",
        "universal-credit",
        "homelessness-help-from-council",
        "council-tax",
        "apply-council-tax-reduction",
    ],
    "health": [
        "help-nhs-costs",
        "healthcare-immigration-application",
    ],
}

NHS = {
    "health": [
        "https://www.nhs.uk/nhs-services/gps/how-to-register-with-a-gp-surgery/",
        "https://www.nhs.uk/nhs-services/gps/gp-appointments-and-bookings/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-use-111/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-go-to-ae/",
        "https://www.nhs.uk/nhs-services/urgent-and-emergency-care-services/when-to-call-999/",
        "https://www.nhs.uk/nhs-services/dentists/how-to-find-an-nhs-dentist/",
        "https://www.nhs.uk/nhs-services/dentists/how-much-nhs-dental-treatment-costs/",
        "https://www.nhs.uk/nhs-services/pharmacies/how-pharmacies-can-help/",
        "https://www.nhs.uk/nhs-services/help-with-health-costs/nhs-low-income-scheme-lis/",
        "https://www.nhs.uk/nhs-services/visiting-or-moving-to-england/how-to-access-nhs-services-in-england-if-you-are-visiting-from-abroad/",
        "https://www.nhs.uk/vaccinations/nhs-vaccinations-and-when-to-have-them/",
        "https://www.nhs.uk/pregnancy/finding-out/what-to-do-when-you-find-out-you-are-pregnant/",
    ],
}


In [ ]:
%%writefile src/ingest.py
"""Fetch every source page, keep a raw snapshot, and write one clean record per page.

Output: data/raw/<id>.(json|html) and data/processed/pages.jsonl
Each record keeps the section structure (h2/h3 headings) so chunks can carry
their heading, plus three dates: the publisher's own date, the publisher's
last technical update, and when we fetched it (for data-currency reporting).
"""
import hashlib
import json
import pathlib
import time
from datetime import datetime, timezone

import requests
from bs4 import BeautifulSoup

from sources import GOVUK, NHS

ROOT = pathlib.Path(__file__).resolve().parents[1]
RAW = ROOT / "data" / "raw"
OUT = ROOT / "data" / "processed" / "pages.jsonl"
UA = {"User-Agent": "MSc research prototype (multilingual civic assistant); polite crawler"}

# GOV.UK formats store text in different fields; order follows the rendered page.
GOVUK_TEXT_FIELDS = ["introduction", "introductory_paragraph", "body", "more_information",
                     "need_to_know", "other_ways_to_apply", "what_you_need_to_know"]


def page_id(url):
    return hashlib.sha1(url.encode()).hexdigest()[:10]


def sections_from_html(html, page_title):
    """Split HTML into (heading, text) sections at h2/h3 boundaries."""
    soup = BeautifulSoup(html, "html.parser")
    sections, heading, buf = [], page_title, []
    for el in soup.find_all(["h2", "h3", "p", "li", "td", "th"]):
        if el.name in ("h2", "h3"):
            if buf:
                sections.append({"heading": heading, "text": " ".join(buf)})
            heading, buf = el.get_text(" ", strip=True), []
        elif not el.find_parent(["li", "td", "th"]) or el.name in ("td", "th"):
            # skip nested elements already covered by their parent's text
            t = el.get_text(" ", strip=True)
            if t:
                buf.append(t)
    if buf:
        sections.append({"heading": heading, "text": " ".join(buf)})
    return sections


def fetch_govuk(path, domain):
    r = requests.get("https://www.gov.uk/api/content/" + path, headers=UA, timeout=30)
    r.raise_for_status()
    data = r.json()
    url = "https://www.gov.uk/" + path
    (RAW / f"{page_id(url)}.json").write_text(r.text)
    det = data.get("details", {})
    html = "".join(det.get(f, "") for f in GOVUK_TEXT_FIELDS if isinstance(det.get(f), str))
    for part in det.get("parts", []):
        html += f"<h2>{part['title']}</h2>{part['body']}"
    title = data.get("title", "")
    return {
        "id": page_id(url), "url": url, "source": "GOV.UK", "domain": domain,
        "title": title, "format": data.get("schema_name"),
        "public_updated_at": (data.get("public_updated_at") or "")[:10],
        "updated_at": (data.get("updated_at") or "")[:10],
        "sections": sections_from_html(html, title),
    }


def fetch_nhs(url, domain):
    r = requests.get(url, headers=UA, timeout=30)
    r.raise_for_status()
    (RAW / f"{page_id(url)}.html").write_text(r.text)
    soup = BeautifulSoup(r.text, "html.parser")
    reviewed = soup.find(string=lambda s: s and "Page last reviewed" in s)
    reviewed = reviewed.strip().split(":", 1)[1].strip() if reviewed else ""
    modified = ""
    for s in soup.find_all("script", type="application/ld+json"):
        try:
            modified = (json.loads(s.string or "{}").get("dateModified") or modified)[:10]
        except (ValueError, AttributeError):
            pass
    article = soup.find("article") or soup.find("main")
    for junk in article.find_all(["aside", "nav", "footer", "script", "style"]):
        junk.decompose()
    title = soup.find("h1").get_text(" ", strip=True)
    # the lead sentence ("999 is for life-threatening emergencies...") sits outside <article>
    lede = soup.find(class_="nhsuk-lede-text")
    html = (str(lede) if lede and article not in lede.parents else "") + str(article)
    return {
        "id": page_id(url), "url": url, "source": "NHS.uk", "domain": domain,
        "title": title, "format": "nhs_article",
        "public_updated_at": datetime.strptime(reviewed, "%d %B %Y").strftime("%Y-%m-%d") if reviewed else "",
        "updated_at": modified,
        "sections": sections_from_html(html, title),
    }


def main():
    RAW.mkdir(parents=True, exist_ok=True)
    OUT.parent.mkdir(parents=True, exist_ok=True)
    fetched_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
    jobs = [(fetch_govuk, p, d) for d, ps in GOVUK.items() for p in ps]
    jobs += [(fetch_nhs, u, d) for d, us in NHS.items() for u in us]
    records, failed = [], []
    for fn, target, domain in jobs:
        try:
            rec = fn(target, domain)
            rec["fetched_at"] = fetched_at
            rec["words"] = sum(len(s["text"].split()) for s in rec["sections"])
            records.append(rec)
            print(f"ok   {rec['domain']:8} {rec['words']:5}w  {rec['title']}")
        except Exception as e:  # keep going; report at the end
            failed.append((target, repr(e)))
            print(f"FAIL {target}: {e}")
        time.sleep(0.5)
    with OUT.open("w") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"\n{len(records)} pages written to {OUT.relative_to(ROOT)}; {len(failed)} failed")
    for t, e in failed:
        print("  failed:", t, e)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/chunk.py
"""Turn pages into retrievable chunks that respect the page's own sections.

Rules (kept simple so they are easy to describe in the methodology):
  * one chunk per h2/h3 section;
  * each section is tagged with the UK nation it applies to (from its heading);
  * sections shorter than MIN_WORDS are merged into the next one on the same page,
    but only when both have the same nation tag;
  * sections longer than MAX_WORDS are split into windows with OVERLAP words;
  * every chunk text starts with "<page title> - <section heading>:" so the
    embedding sees the context a reader would see.
Output: data/processed/chunks.csv
"""
import json
import pathlib
import re

import pandas as pd

ROOT = pathlib.Path(__file__).resolve().parents[1]
MIN_WORDS, MAX_WORDS, OVERLAP = 40, 200, 40
OTHER_NATIONS = ["Scotland", "Wales", "Northern Ireland"]


def nation_of(heading):
    """GOV.UK marks nation-specific sections in the heading ("Scotland", "If you're in Wales...").
    Everything else applies to England (or the whole UK). NHS.uk pages cover England only."""
    named = [n for n in OTHER_NATIONS if re.search(rf"\b{n}\b", heading)]
    return ";".join(n.lower().replace(" ", "_") for n in named) or "england"


def join(a, b):
    return {"heading": a["heading"] + " / " + b["heading"], "text": a["text"] + " " + b["text"],
            "nation": a["nation"]}


def merge_small(sections):
    merged, carry = [], None
    for s in sections:
        s = {**s, "nation": nation_of(s["heading"])}
        if carry and carry["nation"] == s["nation"]:
            s = join(carry, s)
        elif carry:
            merged.append(carry)  # never merge across nations; keep the short one alone
        carry = None
        if len(s["text"].split()) < MIN_WORDS:
            carry = s
        else:
            merged.append(s)
    if carry:  # trailing short section joins the previous one if same nation, or stands alone
        if merged and merged[-1]["nation"] == carry["nation"]:
            merged[-1] = join(merged[-1], carry)
        else:
            merged.append(carry)
    return merged


def windows(words):
    if len(words) <= MAX_WORDS:
        return [words]
    step = MAX_WORDS - OVERLAP
    return [words[i:i + MAX_WORDS] for i in range(0, len(words) - OVERLAP, step)]


def main():
    pages = [json.loads(l) for l in open(ROOT / "data/processed/pages.jsonl")]
    rows = []
    for p in pages:
        for s_i, s in enumerate(merge_small(p["sections"])):
            for w_i, w in enumerate(windows(s["text"].split())):
                body = " ".join(w)
                rows.append({
                    "chunk_id": f"{p['id']}-{s_i:02d}-{w_i}",
                    "page_id": p["id"], "url": p["url"], "source": p["source"], "domain": p["domain"],
                    "title": p["title"], "heading": s["heading"], "nation": s["nation"],
                    "public_updated_at": p["public_updated_at"], "fetched_at": p["fetched_at"],
                    "words": len(w), "body": body,
                    "text": f"{p['title']} - {s['heading']}: {body}",
                })
    df = pd.DataFrame(rows)
    df.to_csv(ROOT / "data/processed/chunks.csv", index=False)
    print(f"{len(pages)} pages -> {len(df)} chunks")
    print(df.groupby("domain").agg(pages=("page_id", "nunique"), chunks=("chunk_id", "count"),
                                   words=("words", "sum")).to_string())
    print("nation tags:", df.nation.value_counts().to_dict())
    print("chunk words: median", int(df.words.median()), "min", df.words.min(), "max", df.words.max())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/retrieval.py
"""Dense and lexical retrievers over the chunk table.

Dense: sentence-transformers model + FAISS inner-product index on normalised
vectors (= cosine similarity). E5 models need "query: " / "passage: " prefixes.
Lexical: BM25 (rank_bm25), used as a non-neural baseline.
"""
import re

import torch  # noqa: F401  must load before faiss: on macOS the two OpenMP runtimes otherwise crash NLLB generation
import faiss
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer


class DenseRetriever:
    def __init__(self, model_name, texts):
        self.model = SentenceTransformer(model_name)
        self.e5 = "e5" in model_name.lower()
        passages = [("passage: " + t) if self.e5 else t for t in texts]
        emb = self.model.encode(passages, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
        self.index = faiss.IndexFlatIP(emb.shape[1])
        self.index.add(np.asarray(emb, dtype="float32"))

    def search(self, query, k=10):
        q = self.model.encode([("query: " + query) if self.e5 else query], normalize_embeddings=True)
        scores, ids = self.index.search(np.asarray(q, dtype="float32"), k)
        return list(zip(ids[0].tolist(), scores[0].tolist()))

    def save(self, path):
        faiss.write_index(self.index, str(path))


def tokenize(text):
    return re.findall(r"\w+", text.lower())


class BM25Retriever:
    def __init__(self, texts):
        self.bm25 = BM25Okapi([tokenize(t) for t in texts])

    def search(self, query, k=10):
        scores = self.bm25.get_scores(tokenize(query))
        ids = np.argsort(-scores)[:k]
        return [(int(i), float(scores[i])) for i in ids]


In [ ]:
%%writefile src/language.py
"""Detect which supported language a question is written in.

Uses lingua (Apache-2.0), restricted to the five project languages, which makes
short questions much easier to classify than open-set detection. Arabic and Urdu
share a script, so restricting the set matters most for those two.
"""
from lingua import Language, LanguageDetectorBuilder

SUPPORTED = {
    Language.ENGLISH: "en", Language.POLISH: "pl", Language.ROMANIAN: "ro",
    Language.ARABIC: "ar", Language.URDU: "ur",
}
_detector = LanguageDetectorBuilder.from_languages(*SUPPORTED).build()


def detect(text):
    """Return 'en', 'pl', 'ro', 'ar' or 'ur'; falls back to 'en' if lingua cannot decide."""
    lang = _detector.detect_language_of(text)
    return SUPPORTED.get(lang, "en")


In [ ]:
%%writefile src/translate.py
"""Machine translation with NLLB-200 (facebook/nllb-200-distilled-600M, CC-BY-NC-4.0).

Used in week 5 to translate questions into English before retrieval, and in week 6
for answers going back into the user's language. Non-commercial licence: fine for
this research prototype, not for a production service.
"""
import re

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL = "facebook/nllb-200-distilled-600M"
NLLB_CODES = {"en": "eng_Latn", "pl": "pol_Latn", "ro": "ron_Latn", "ar": "arb_Arab", "ur": "urd_Arab"}


class Translator:
    def __init__(self, model_name=MODEL):
        self.tok = AutoTokenizer.from_pretrained(model_name)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"  # GPU on Colab, CPU elsewhere
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).eval().to(self.device)

    @torch.inference_mode()
    def _translate_sentences(self, sentences, src, tgt, max_length):
        self.tok.src_lang = NLLB_CODES[src]
        batch = self.tok(sentences, return_tensors="pt", padding=True, truncation=True, max_length=400).to(self.device)
        out = self.model.generate(
            **batch, max_length=max_length, num_beams=4,
            forced_bos_token_id=self.tok.convert_tokens_to_ids(NLLB_CODES[tgt]))
        return self.tok.batch_decode(out, skip_special_tokens=True)

    def translate(self, texts, src, tgt="en", max_length=256):
        """Translate a list of strings from language `src` to `tgt` (project codes, e.g. 'pl').

        NLLB is trained on single sentences and silently drops later sentences of a
        multi-sentence input, so each text is split into sentences first and rejoined.
        """
        if src == tgt:
            return list(texts)
        split = [split_sentences(t) for t in texts]
        flat = [s for parts in split for s in parts]
        done = iter(self._translate_sentences(flat, src, tgt, max_length))
        return [" ".join(next(done) for _ in parts) for parts in split]


def split_sentences(text):
    """Split on sentence-final punctuation, including Arabic ؟ and Urdu ۔ ."""
    parts = [p.strip() for p in re.split(r"(?<=[.!?؟۔])\s+", text.strip())]
    return [p for p in parts if p] or [text]


In [ ]:
%%writefile src/rerank.py
"""Second-stage reranking with a multilingual cross-encoder (BAAI/bge-reranker-v2-m3, Apache-2.0).

The bi-encoder (e5) compares a question vector with pre-computed chunk vectors, which
is fast but coarse. A cross-encoder reads the question and one chunk together and
scores how well that chunk answers it; it is too slow for the whole corpus, so it
only reorders the top candidates from the first stage.
"""
from sentence_transformers import CrossEncoder

MODEL = "BAAI/bge-reranker-v2-m3"


class Reranker:
    def __init__(self, model_name=MODEL):
        self.model = CrossEncoder(model_name, max_length=512)

    def rerank(self, query, candidates, texts):
        """candidates: list of chunk ids; texts: chunk id -> text. Returns [(id, score)] best first."""
        if not candidates:
            return []
        scores = self.model.predict([(query, texts[i]) for i in candidates], batch_size=16, show_progress_bar=False)
        return sorted(zip(candidates, map(float, scores)), key=lambda x: -x[1])


In [ ]:
%%writefile src/pipeline.py
"""Week 5 search pipeline: question in any supported language -> best official passages.

    question
      -> detect language                       (language.py)
      -> translate to English if not English   (translate.py, optional)
      -> dense retrieval, original + English   (retrieval.py, e5 + FAISS)
      -> fuse the two ranked lists (RRF)
      -> keep England / UK-wide chunks only     (nation tag from chunk.py)
      -> rerank top candidates                  (rerank.py, optional)
      -> top k passages with title, heading, URL and dates for citation
"""
import pathlib

import pandas as pd

from language import detect
from retrieval import DenseRetriever

ROOT = pathlib.Path(__file__).resolve().parents[1]
E5 = "intfloat/multilingual-e5-base"
RRF_K = 60  # standard constant from Cormack et al. (2009)


def rrf(*ranked_lists, k=RRF_K):
    """Reciprocal rank fusion: combine ranked lists of ids into one ranking."""
    score = {}
    for ranked in ranked_lists:
        for rank, i in enumerate(ranked, start=1):
            score[i] = score.get(i, 0.0) + 1.0 / (k + rank)
    return sorted(score, key=lambda i: -score[i])


class SearchPipeline:
    def __init__(self, chunks=None, translate=True, rerank=True, england_only=True, pool=20):
        self.chunks = chunks if chunks is not None else pd.read_csv(ROOT / "data/processed/chunks.csv")
        self.texts = self.chunks.text.tolist()
        self.retriever = DenseRetriever(E5, self.texts)
        self.allowed = set(self.chunks.index[self.chunks.nation == "england"]) if england_only else None
        self.pool = pool
        self.translator = None
        if translate:
            from translate import Translator
            self.translator = Translator()
        self.reranker = None
        if rerank:
            from rerank import Reranker
            self.reranker = Reranker()

    def _dense(self, query, n):
        return [i for i, _ in self.retriever.search(query, k=n)]

    def search(self, question, k=3, lang=None):
        lang = lang or detect(question)
        english = question
        lists = [self._dense(question, 50)]
        if self.translator and lang != "en":
            english = self.translator.translate([question], src=lang)[0]
            lists.append(self._dense(english, 50))
        ranked = rrf(*lists) if len(lists) > 1 else lists[0]
        if self.allowed is not None:
            ranked = [i for i in ranked if i in self.allowed]
        ranked = ranked[:self.pool]
        scores = {}
        if self.reranker:
            scores = dict(self.reranker.rerank(question, ranked, self.texts))
            ranked = sorted(ranked, key=lambda i: -scores[i])
        results = []
        for i in ranked[:k]:
            c = self.chunks.iloc[i]
            results.append({"chunk": int(i), "title": c.title, "heading": c.heading, "url": c.url,
                            "source": c.source, "nation": c.nation, "public_updated_at": c.public_updated_at,
                            "fetched_at": c.fetched_at, "text": c.body,
                            "score": scores.get(i)})  # reranker relevance (0-1), used by the refusal rule
        return {"lang": lang, "english_query": english, "results": results}


In [ ]:
%%writefile src/generate.py
"""Grounded answer generation with an openly licensed LLM (Qwen/Qwen3.5-4B, Apache-2.0).

Why this model (task T26):
  * Apache-2.0 and not gated, so it can be run locally or on a free Colab GPU and user questions
    never leave the machine (see docs/ethics-notes.md: no third-party API without consent).
  * The model card states support for 201 languages and dialects; Polish, Romanian, Arabic and Urdu
    are all needed here.
  * 4B parameters (9.3 GB of weights in 16-bit): fits a 16 GB Colab T4 in float16 and the 24 GB development laptop in bfloat16.
Qwen3.5 "thinks" before answering by default; thinking is switched off (enable_thinking=False) so the
output is the answer only. Decoding is greedy so that every run gives the same answer.
"""
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen3.5-4B"
NO_ANSWER = "NO_ANSWER"
LANGUAGE_NAMES = {"en": "English", "pl": "Polish", "ro": "Romanian", "ar": "Arabic", "ur": "Urdu"}

SYSTEM = """You help people in England understand official guidance on health services, housing and schools.
You are given numbered extracts from official GOV.UK and NHS.uk pages, and a question.

Rules:
1. Use ONLY facts stated in the extracts. Never add facts, dates, places or general remarks from your own knowledge.
2. Every sentence must end with the number of the extract it comes from, in square brackets, for example [1] or [1][2].
   If you cannot point to an extract for a sentence, leave that sentence out.
3. Answer the question first. Write in plain language: short sentences, everyday words, no more than 120 words.
4. If the extracts do not answer the question, reply with exactly NO_ANSWER and nothing else.
5. Do not give medical diagnoses or legal advice about the user's own case; explain what the guidance says.
6. Write the whole answer in {language}."""

CHECK = """You check whether numbered extracts from official GOV.UK and NHS.uk pages answer a question.
The question may be in any language. Reply YES if the extracts contain the information needed to answer it,
even partly. Reply NO if they do not. Reply with one word: YES or NO."""

CITATION = re.compile(r"\[(\d+)\]")


def format_sources(passages):
    """Number the passages [1]..[k] the way the model will cite them."""
    return "\n\n".join(f"[{n}] {p['title']} - {p['heading']}\n{p['text']}" for n, p in enumerate(passages, 1))


def build_messages(question, passages, answer_lang, pivot=False):
    """pivot=True: the answer is written in English for a question in another language (translate mode).
    The model tends to follow the question's language, so the instruction is repeated after the question."""
    system = SYSTEM.format(language=LANGUAGE_NAMES[answer_lang])
    user = f"Extracts:\n\n{format_sources(passages)}\n\nQuestion: {question}"
    if pivot:
        user += "\n\nWrite the answer in English, even though the question is in another language."
    return [{"role": "system", "content": system}, {"role": "user", "content": user}]


def check_messages(question, passages):
    user = f"Extracts:\n\n{format_sources(passages)}\n\nQuestion: {question}\n\nDo the extracts answer the question?"
    return [{"role": "system", "content": CHECK}, {"role": "user", "content": user}]


def citations(answer):
    """Citation numbers used in an answer, in order of first use."""
    return list(dict.fromkeys(int(n) for n in CITATION.findall(answer)))


class Generator:
    def __init__(self, model_name=MODEL, device=None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
        dtype = torch.float16 if device == "cuda" else torch.bfloat16 if device == "mps" else torch.float32
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype).to(device).eval()

    @torch.inference_mode()
    def chat(self, messages, max_new_tokens=300):
        prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True,
                                                    enable_thinking=False)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.device)
        out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        return self.tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    def answerable(self, question, passages):
        """Refusal rule 2: a separate yes/no call - do the passages answer the question?"""
        return self.chat(check_messages(question, passages), max_new_tokens=3).strip().upper().startswith("YES")

    def answer(self, question, passages, answer_lang, pivot=False):
        """Answer from the passages only. Returns the raw model text (may be NO_ANSWER)."""
        return self.chat(build_messages(question, passages, answer_lang, pivot))


In [ ]:
%%writefile src/assistant.py
"""Week 6 assistant: question in any supported language -> cited answer from official pages, or a refusal.

    question
      -> search stage (pipeline.py, set-up F from week 5): top 3 passages + reranker score
      -> refusal rule 1: top reranker score below THRESHOLD -> refuse without calling the LLM
      -> refusal rule 2: a separate yes/no LLM call says the passages do not answer it -> refuse
      -> LLM answers only from the 3 passages, citing [n]          (generate.py)
      -> refusal rule 3: the LLM replies NO_ANSWER -> refuse
      -> answer language: "direct"    = the LLM writes in the user's language
                          "translate" = the LLM writes in English, NLLB translates it
      -> source cards for the cited passages (title, section, link, date fetched)
      -> health note (NHS 111 / 999) whenever an NHS.uk page is cited

Citations are numbers chosen by the LLM; titles, links and dates always come from the corpus, so the
model can never invent a URL.
"""
from generate import NO_ANSWER, Generator, citations, CITATION
from pipeline import SearchPipeline
from translate import split_sentences

# Calibrated in eval/evaluate_week6.py on 150 answerable and 50 unanswerable questions.
THRESHOLD = 0.0052  # 0.9 x lowest top score of an answerable question whose answer was retrieved

# Fixed messages (written by the author; part of the native-speaker check, task T22).
REFUSAL = {
    "en": "I could not find an answer to this in the official GOV.UK and NHS pages I use. "
          "Please check GOV.UK or NHS.uk, or contact your local council.",
    "pl": "Nie udało mi się znaleźć odpowiedzi na to pytanie na oficjalnych stronach GOV.UK i NHS. "
          "Sprawdź GOV.UK lub NHS.uk albo skontaktuj się z lokalnym urzędem (council).",
    "ro": "Nu am găsit un răspuns la această întrebare în paginile oficiale GOV.UK și NHS. "
          "Vă rugăm să verificați GOV.UK sau NHS.uk ori să contactați consiliul local.",
    "ar": "لم أجد إجابة عن هذا السؤال في صفحات GOV.UK وNHS الرسمية. "
          "يُرجى مراجعة GOV.UK أو NHS.uk، أو التواصل مع المجلس المحلي.",
    "ur": "مجھے سرکاری GOV.UK اور NHS صفحات میں اس سوال کا جواب نہیں ملا۔ "
          "براہ کرم GOV.UK یا NHS.uk دیکھیں، یا اپنی مقامی کونسل سے رابطہ کریں۔",
}
HEALTH_NOTE = {
    "en": "If you need medical help now, call 111. In an emergency, call 999.",
    "pl": "Jeśli potrzebujesz pilnej pomocy medycznej, zadzwoń pod numer 111. W nagłym wypadku dzwoń pod 999.",
    "ro": "Dacă aveți nevoie urgent de ajutor medical, sunați la 111. În caz de urgență, sunați la 999.",
    "ar": "إذا كنت بحاجة إلى مساعدة طبية الآن، اتصل بالرقم 111. في حالات الطوارئ، اتصل بالرقم 999.",
    "ur": "اگر آپ کو ابھی طبی مدد کی ضرورت ہے تو 111 پر کال کریں۔ ہنگامی صورت میں 999 پر کال کریں۔",
}


def translate_answer(translator, english, lang):
    """Translate an English answer sentence by sentence, keeping each sentence's [n] citations.

    Citation markers are taken out before translation (NLLB may drop or garble them) and put back
    at the end of the translated sentence."""
    out = []
    for sent in split_sentences(english):
        marks = "".join(f"[{n}]" for n in CITATION.findall(sent))
        text = CITATION.sub("", sent).replace(" .", ".").strip()
        if text:
            out.append((translator.translate([text], src="en", tgt=lang)[0] + " " + marks).strip())
        elif marks and out:  # markers split off on their own belong to the previous sentence
            out[-1] += " " + marks
    return " ".join(out)


class Assistant:
    def __init__(self, pipeline=None, generator=None, threshold=THRESHOLD, mode="direct"):
        self.pipeline = pipeline or SearchPipeline()
        self.generator = generator or Generator()
        self.threshold = threshold
        self.mode = mode

    def refuse(self, base, reason):
        return {**base, "refused": True, "reason": reason, "answer": REFUSAL[base["lang"]], "sources": []}

    def answer_from(self, search, question, mode=None):
        """Answer from an existing search result (lets the evaluation reuse one search for both modes)."""
        mode = mode or self.mode
        lang, passages = search["lang"], search["results"]
        top = passages[0]["score"] if passages else 0.0
        base = {"question": question, "lang": lang, "mode": mode, "english_query": search["english_query"],
                "top_score": top, "raw": ""}
        if top < self.threshold:
            return self.refuse(base, "low relevance")
        if not self.generator.answerable(question, passages):
            return self.refuse(base, "check: not in sources")
        answer_lang = lang if mode == "direct" else "en"
        raw = self.generator.answer(question, passages, answer_lang, pivot=answer_lang != lang)
        base["raw"] = raw
        if NO_ANSWER in raw:
            return self.refuse(base, "model: not in sources")
        text = raw if answer_lang == lang else translate_answer(self.pipeline.translator, raw, lang)
        cited = [n for n in citations(text) if 1 <= n <= len(passages)]
        sources = [{"n": n, **{k: passages[n - 1][k] for k in ("title", "heading", "url", "source", "fetched_at")}}
                   for n in cited]
        if any(s["source"] == "NHS.uk" for s in sources):
            text += "\n\n" + HEALTH_NOTE[lang]
        return {**base, "refused": False, "reason": None, "answer": text, "sources": sources,
                "passages": passages}

    def ask(self, question, lang=None, mode=None):
        return self.answer_from(self.pipeline.search(question, k=3, lang=lang), question, mode)


In [ ]:
%%writefile eval/build_queries.py
"""Build the retrieval test set and check every answer phrase against the corpus.

Each English question has:
  gold_url - the page that answers it
  evidence - a short phrase copied verbatim from that page that contains the answer
A retrieved chunk only counts as "answer-bearing" if it contains the evidence
phrase, so a hit on the right page but the wrong section is not counted.

Translations (pl, ro, ar, ur) were written by the author and are flagged
"unverified" until a native speaker checks them.
Output: eval/queries.csv
"""
import csv
import json
import pathlib
import sys

ROOT = pathlib.Path(__file__).resolve().parents[1]
G = "https://www.gov.uk/"
N = "https://www.nhs.uk/nhs-services/"

# qid: (domain, gold_url, evidence, {lang: question})
Q = {
    "S01": ("schools", G + "schools-admissions", "close on 15 January", {
        "en": "When is the deadline to apply for a primary school place?",
        "pl": "Jaki jest termin składania wniosku o miejsce w szkole podstawowej?",
        "ro": "Care este termenul limită pentru a depune cererea pentru un loc la școala primară?",
        "ar": "ما هو الموعد النهائي للتقديم على مقعد في المدرسة الابتدائية؟",
        "ur": "پرائمری اسکول میں داخلے کی درخواست دینے کی آخری تاریخ کیا ہے؟"}),
    "S02": ("schools", G + "schools-admissions", "by 31 October", {
        "en": "What is the last date to apply for a secondary school place?",
        "pl": "Do kiedy trzeba złożyć wniosek o miejsce w szkole średniej?",
        "ro": "Până la ce dată trebuie să aplic pentru un loc la școala secundară?",
        "ar": "ما هو آخر موعد للتقديم على مقعد في المدرسة الثانوية؟",
        "ur": "سیکنڈری اسکول میں داخلے کے لیے درخواست کس تاریخ تک دینی ہوگی؟"}),
    "S03": ("schools", G + "schools-admissions", "primary schools on 16 April", {
        "en": "When will I find out which primary school my child has been offered?",
        "pl": "Kiedy dowiem się, do której szkoły podstawowej zostało przyjęte moje dziecko?",
        "ro": "Când voi afla la ce școală primară a primit loc copilul meu?",
        "ar": "متى سأعرف المدرسة الابتدائية التي قُبل فيها طفلي؟",
        "ur": "مجھے کب پتہ چلے گا کہ میرے بچے کو کون سے پرائمری اسکول میں جگہ ملی ہے؟"}),
    "S04": ("schools", G + "schools-admissions", "at least 20 school days", {
        "en": "How long do I have to appeal if my child is refused a school place?",
        "pl": "Ile mam czasu na odwołanie, jeśli moje dziecko nie dostało miejsca w szkole?",
        "ro": "Cât timp am la dispoziție să fac contestație dacă copilul meu nu a primit loc la școală?",
        "ar": "كم لدي من الوقت للاستئناف إذا رُفض طلب طفلي للحصول على مقعد في المدرسة؟",
        "ur": "اگر میرے بچے کو اسکول میں جگہ نہ ملے تو اپیل کرنے کے لیے میرے پاس کتنا وقت ہے؟"}),
    "S05": ("schools", G + "school-attendance-absence", "fined or face legal action for taking your child on holiday", {
        "en": "Can I be fined for taking my child on holiday during term time?",
        "pl": "Czy mogę dostać grzywnę za zabranie dziecka na wakacje w trakcie roku szkolnego?",
        "ro": "Pot primi amendă dacă îmi iau copilul în vacanță în timpul semestrului?",
        "ar": "هل يمكن أن أُغرَّم إذا أخذت طفلي في عطلة أثناء الفصل الدراسي؟",
        "ur": "کیا ٹرم کے دوران بچے کو چھٹیوں پر لے جانے پر مجھے جرمانہ ہو سکتا ہے؟"}),
    "S06": ("schools", G + "apply-free-school-meals", "free school meals if you get any of the following", {
        "en": "Can my child get free school meals if I receive Universal Credit?",
        "pl": "Czy moje dziecko może dostać darmowe posiłki w szkole, jeśli otrzymuję Universal Credit?",
        "ro": "Poate copilul meu să primească mese gratuite la școală dacă primesc Universal Credit?",
        "ar": "هل يمكن لطفلي الحصول على وجبات مدرسية مجانية إذا كنت أتلقى Universal Credit؟",
        "ur": "اگر مجھے Universal Credit ملتا ہے تو کیا میرے بچے کو اسکول میں مفت کھانا مل سکتا ہے؟"}),
    "S07": ("schools", G + "free-school-transport", "more than 2 miles away and the child is under 8", {
        "en": "Is school transport free if the school is far from home?",
        "pl": "Czy transport do szkoły jest darmowy, jeśli szkoła jest daleko od domu?",
        "ro": "Este transportul la școală gratuit dacă școala este departe de casă?",
        "ar": "هل المواصلات المدرسية مجانية إذا كانت المدرسة بعيدة عن المنزل؟",
        "ur": "اگر اسکول گھر سے دور ہو تو کیا اسکول کی ٹرانسپورٹ مفت ہے؟"}),
    "S08": ("schools", G + "know-when-you-can-leave-school", "16 by the end of the summer holidays", {
        "en": "At what age can my child leave school?",
        "pl": "W jakim wieku moje dziecko może skończyć szkołę?",
        "ro": "La ce vârstă poate copilul meu să termine școala?",
        "ar": "في أي عمر يمكن لطفلي ترك المدرسة؟",
        "ur": "میرا بچہ کس عمر میں اسکول چھوڑ سکتا ہے؟"}),
    "S09": ("schools", G + "children-with-special-educational-needs", "SEN support - support given in school", {
        "en": "What support can my child get for special educational needs?",
        "pl": "Jakie wsparcie może otrzymać moje dziecko ze specjalnymi potrzebami edukacyjnymi?",
        "ro": "Ce sprijin poate primi copilul meu pentru cerințe educaționale speciale?",
        "ar": "ما الدعم الذي يمكن أن يحصل عليه طفلي إذا كانت لديه احتياجات تعليمية خاصة؟",
        "ur": "خصوصی تعلیمی ضروریات والے میرے بچے کو کیا مدد مل سکتی ہے؟"}),
    "S10": ("schools", G + "complain-about-school", "different ways to complain in England", {
        "en": "How do I complain about my child's state school?",
        "pl": "Jak złożyć skargę na państwową szkołę mojego dziecka?",
        "ro": "Cum fac o plângere împotriva școlii de stat a copilului meu?",
        "ar": "كيف أقدم شكوى ضد المدرسة الحكومية التي يدرس فيها طفلي؟",
        "ur": "میں اپنے بچے کے سرکاری اسکول کے بارے میں شکایت کیسے کروں؟"}),

    "H01": ("housing", G + "guidance/renters-rights-act-overview-for-tenants", "section 21 process", {
        "en": "Can my landlord evict me without giving a reason?",
        "pl": "Czy wynajmujący może mnie eksmitować bez podania powodu?",
        "ro": "Poate proprietarul să mă evacueze fără să-mi dea un motiv?",
        "ar": "هل يمكن للمالك أن يطردني دون إبداء سبب؟",
        "ur": "کیا مالک مکان مجھے بغیر کوئی وجہ بتائے گھر سے نکال سکتا ہے؟"}),
    "H02": ("housing", G + "guidance/renters-rights-act-overview-for-tenants", "increase the rent once a year", {
        "en": "How often can my landlord increase the rent?",
        "pl": "Jak często wynajmujący może podnosić czynsz?",
        "ro": "Cât de des poate proprietarul să crească chiria?",
        "ar": "كم مرة يمكن للمالك أن يرفع الإيجار؟",
        "ur": "مالک مکان کتنی بار کرایہ بڑھا سکتا ہے؟"}),
    "H03": ("housing", G + "tenancy-deposit-protection", "Deposit Protection Service", {
        "en": "My landlord took a deposit from me. Does it have to be protected?",
        "pl": "Wynajmujący pobrał ode mnie kaucję. Czy musi ona być zabezpieczona?",
        "ro": "Proprietarul mi-a luat o garanție. Trebuie aceasta să fie protejată?",
        "ar": "أخذ المالك مني مبلغ تأمين، هل يجب أن يكون هذا التأمين محميًا؟",
        "ur": "مالک مکان نے مجھ سے ڈپازٹ لیا ہے، کیا اس ڈپازٹ کا محفوظ ہونا ضروری ہے؟"}),
    "H04": ("housing", G + "housing-benefit", "help you pay your rent", {
        "en": "How can I get help paying my rent if I am on a low income?",
        "pl": "Jak uzyskać pomoc w opłaceniu czynszu, jeśli mam niskie dochody?",
        "ro": "Cum pot primi ajutor pentru plata chiriei dacă am un venit mic?",
        "ar": "كيف أحصل على مساعدة في دفع الإيجار إذا كان دخلي منخفضًا؟",
        "ur": "کم آمدنی کی صورت میں مجھے کرایہ ادا کرنے میں مدد کیسے مل سکتی ہے؟"}),
    "H05": ("housing", G + "council-housing", "join a waiting list", {
        "en": "How do I apply for council housing?",
        "pl": "Jak złożyć wniosek o mieszkanie komunalne?",
        "ro": "Cum aplic pentru o locuință socială de la consiliul local?",
        "ar": "كيف أتقدم بطلب للحصول على سكن من المجلس المحلي؟",
        "ur": "میں کونسل کے مکان کے لیے درخواست کیسے دوں؟"}),
    "H06": ("housing", G + "homelessness-help-from-council", "homeless or about to lose your home", {
        "en": "I am about to lose my home. Can the council help me?",
        "pl": "Zaraz stracę dom. Czy lokalny urząd może mi pomóc?",
        "ro": "Sunt pe punctul de a-mi pierde locuința. Mă poate ajuta consiliul local?",
        "ar": "أنا على وشك أن أفقد منزلي، هل يمكن للمجلس المحلي مساعدتي؟",
        "ur": "میرا گھر چھننے والا ہے، کیا کونسل میری مدد کر سکتی ہے؟"}),
    "H07": ("housing", G + "private-renting", "safe and in a good state of repair", {
        "en": "Does my landlord have to keep my rented home in good repair?",
        "pl": "Czy wynajmujący musi utrzymywać wynajmowane mieszkanie w dobrym stanie technicznym?",
        "ro": "Este proprietarul obligat să mențină locuința închiriată în stare bună?",
        "ar": "هل يجب على المالك أن يحافظ على المنزل المؤجر في حالة جيدة؟",
        "ur": "کیا مالک مکان پر کرائے کے گھر کو اچھی حالت میں رکھنا لازم ہے؟"}),
    "H08": ("housing", G + "private-renting", "24 hours", {
        "en": "How much notice must my landlord give before visiting the property?",
        "pl": "Z jakim wyprzedzeniem wynajmujący musi mnie uprzedzić przed wizytą w mieszkaniu?",
        "ro": "Cu cât timp înainte trebuie să mă anunțe proprietarul dacă vrea să vină în locuință?",
        "ar": "كم من الوقت يجب أن يخطرني المالك قبل أن يزور المنزل؟",
        "ur": "گھر آنے سے پہلے مالک مکان کو مجھے کتنا پہلے اطلاع دینی چاہیے؟"}),
    "H09": ("housing", G + "council-tax", "25% off your bill", {
        "en": "Do I get a council tax discount if I live alone?",
        "pl": "Czy dostanę zniżkę na podatek lokalny, jeśli mieszkam sam?",
        "ro": "Primesc reducere la impozitul local pe locuință dacă locuiesc singur?",
        "ar": "هل أحصل على خصم في ضريبة المجلس المحلي إذا كنت أعيش وحدي؟",
        "ur": "اگر میں اکیلا رہتا ہوں تو کیا مجھے کونسل ٹیکس میں رعایت ملتی ہے؟"}),
    "H10": ("housing", G + "council-housing-association-evictions", "stop or delay the eviction", {
        "en": "What can I do if my council or housing association wants to evict me?",
        "pl": "Co mogę zrobić, jeśli lokalny urząd lub stowarzyszenie mieszkaniowe chce mnie eksmitować?",
        "ro": "Ce pot face dacă consiliul local sau asociația de locuințe vrea să mă evacueze?",
        "ar": "ماذا يمكنني أن أفعل إذا أراد المجلس المحلي أو جمعية الإسكان طردي؟",
        "ur": "اگر کونسل یا ہاؤسنگ ایسوسی ایشن مجھے گھر سے نکالنا چاہے تو میں کیا کر سکتا ہوں؟"}),

    "M01": ("health", N + "gps/how-to-register-with-a-gp-surgery/", "change your current GP surgery online", {
        "en": "How do I register with a GP surgery?",
        "pl": "Jak zarejestrować się u lekarza rodzinnego?",
        "ro": "Cum mă înregistrez la un medic de familie?",
        "ar": "كيف أسجل لدى طبيب عام؟",
        "ur": "میں جی پی کے پاس رجسٹریشن کیسے کرواؤں؟"}),
    "M02": ("health", N + "gps/how-to-register-with-a-gp-surgery/", "do not need ID, proof of address", {
        "en": "Do I need proof of address or immigration status to register with a GP?",
        "pl": "Czy potrzebuję potwierdzenia adresu lub statusu imigracyjnego, aby zarejestrować się u lekarza rodzinnego?",
        "ro": "Am nevoie de dovada adresei sau a statutului de imigrant ca să mă înregistrez la medicul de familie?",
        "ar": "هل أحتاج إلى إثبات عنوان أو إثبات وضع الهجرة للتسجيل لدى طبيب عام؟",
        "ur": "کیا جی پی کے پاس رجسٹر ہونے کے لیے مجھے پتے یا امیگریشن اسٹیٹس کا ثبوت چاہیے؟"}),
    "M03": ("health", N + "urgent-and-emergency-care-services/when-to-use-111/", "if you think you need medical help right now", {
        "en": "When should I use NHS 111?",
        "pl": "Kiedy powinienem skorzystać z NHS 111?",
        "ro": "Când ar trebui să folosesc NHS 111?",
        "ar": "متى يجب أن أستخدم خدمة NHS 111؟",
        "ur": "مجھے NHS 111 کب استعمال کرنا چاہیے؟"}),
    "M04": ("health", N + "urgent-and-emergency-care-services/when-to-go-to-ae/", "serious injuries and life-threatening emergencies", {
        "en": "When should I go to A&E?",
        "pl": "Kiedy powinienem iść na szpitalny oddział ratunkowy?",
        "ro": "Când ar trebui să merg la urgențe?",
        "ar": "متى يجب أن أذهب إلى قسم الطوارئ؟",
        "ur": "مجھے ہسپتال کے ایمرجنسی شعبے میں کب جانا چاہیے؟"}),
    "M05": ("health", N + "urgent-and-emergency-care-services/when-to-call-999/", "999 is for life-threatening emergencies", {
        "en": "When should I call 999?",
        "pl": "Kiedy powinienem dzwonić pod numer 999?",
        "ro": "Când ar trebui să sun la 999?",
        "ar": "متى يجب أن أتصل بالرقم 999؟",
        "ur": "مجھے 999 پر کب کال کرنی چاہیے؟"}),
    "M06": ("health", N + "dentists/how-to-find-an-nhs-dentist/", "Ask for an NHS appointment", {
        "en": "How can I find an NHS dentist?",
        "pl": "Jak znaleźć dentystę NHS?",
        "ro": "Cum găsesc un dentist NHS?",
        "ar": "كيف أجد طبيب أسنان تابعًا لـ NHS؟",
        "ur": "میں NHS کا ڈینٹسٹ کیسے تلاش کروں؟"}),
    "M07": ("health", N + "dentists/how-much-nhs-dental-treatment-costs/", "£27.90", {
        "en": "How much does a basic NHS dental check-up cost?",
        "pl": "Ile kosztuje podstawowy przegląd u dentysty w NHS?",
        "ro": "Cât costă un control stomatologic de bază la NHS?",
        "ar": "كم يكلف فحص الأسنان الأساسي في NHS؟",
        "ur": "NHS میں دانتوں کے بنیادی معائنے پر کتنا خرچ آتا ہے؟"}),
    "M08": ("health", N + "pharmacies/how-pharmacies-can-help/", "This is called Pharmacy First", {
        "en": "Can a pharmacy give me prescription medicine without seeing a GP?",
        "pl": "Czy apteka może wydać mi lek na receptę bez wizyty u lekarza?",
        "ro": "Poate o farmacie să-mi dea medicamente cu rețetă fără să merg la medic?",
        "ar": "هل يمكن للصيدلية أن تعطيني دواءً يحتاج إلى وصفة طبية دون أن أرى الطبيب؟",
        "ur": "کیا فارمیسی ڈاکٹر کو دکھائے بغیر مجھے نسخے والی دوا دے سکتی ہے؟"}),
    "M09": ("health", N + "help-with-health-costs/nhs-low-income-scheme-lis/", "HC2 certificate", {
        "en": "Can I get help with health costs if I have a low income?",
        "pl": "Czy mogę dostać pomoc w pokryciu kosztów leczenia, jeśli mam niskie dochody?",
        "ro": "Pot primi ajutor pentru costurile medicale dacă am venituri mici?",
        "ar": "هل يمكنني الحصول على مساعدة في التكاليف الصحية إذا كان دخلي منخفضًا؟",
        "ur": "کیا کم آمدنی کی صورت میں مجھے علاج کے اخراجات میں مدد مل سکتی ہے؟"}),
    "M10": ("health", G + "healthcare-immigration-application", "£776 per year for students", {
        "en": "How much is the immigration health surcharge for students?",
        "pl": "Ile wynosi dopłata zdrowotna dla imigrantów w przypadku studentów?",
        "ro": "Cât costă taxa de sănătate pentru imigranți în cazul studenților?",
        "ar": "كم تبلغ رسوم الرعاية الصحية للمهاجرين بالنسبة للطلاب؟",
        "ur": "طلبہ کے لیے امیگریشن ہیلتھ سرچارج کتنا ہے؟"}),
}


def main():
    pages = {}
    for line in open(ROOT / "data/processed/pages.jsonl"):
        r = json.loads(line)
        pages[r["url"]] = " ".join(s["text"] for s in r["sections"]).lower()
    bad = [(q, url) for q, (_, url, ev, _) in Q.items() if url not in pages or ev.lower() not in pages[url]]
    if bad:
        sys.exit(f"evidence not found on gold page: {bad}")
    out = ROOT / "eval/queries.csv"
    with out.open("w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["qid", "domain", "lang", "question", "gold_url", "evidence", "translation_status"])
        for qid, (domain, url, ev, qs) in Q.items():
            for lang, text in qs.items():
                w.writerow([qid, domain, lang, text, url, ev, "source" if lang == "en" else "unverified"])
    print(f"{len(Q)} questions x {len(next(iter(Q.values()))[3])} languages -> {out.relative_to(ROOT)}; "
          f"all evidence phrases found verbatim on their gold page")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval/build_unanswerable.py
"""Questions the assistant should refuse: the answer is not in the 38-page corpus.

Each topic was checked against data/processed/chunks.csv (11 Nov 2026). Two are deliberately close to
the corpus: "passport" and "birth certificate" appear only as examples of ID documents, so the search
finds related-looking passages that do not answer the question. U05 asks for medical advice, which is
out of scope. Non-English versions were written by the author (translation_status = unverified, T22).

Output: eval/unanswerable.csv (10 questions x 5 languages = 50 rows)
"""
import csv
import pathlib

ROOT = pathlib.Path(__file__).resolve().parents[1]

QUESTIONS = {
    "U01": {"en": "How do I renew my UK passport?",
            "pl": "Jak odnowić brytyjski paszport?",
            "ro": "Cum îmi reînnoiesc pașaportul britanic?",
            "ar": "كيف أجدد جواز سفري البريطاني؟",
            "ur": "میں اپنا برطانوی پاسپورٹ کیسے تجدید کرواؤں؟"},
    "U02": {"en": "What is the minimum wage for someone aged 21?",
            "pl": "Ile wynosi płaca minimalna dla osoby w wieku 21 lat?",
            "ro": "Care este salariul minim pentru o persoană de 21 de ani?",
            "ar": "ما هو الحد الأدنى للأجور لشخص عمره 21 عامًا؟",
            "ur": "21 سال کے شخص کے لیے کم از کم اجرت کتنی ہے؟"},
    "U03": {"en": "How do I apply for British citizenship?",
            "pl": "Jak ubiegać się o brytyjskie obywatelstwo?",
            "ro": "Cum pot aplica pentru cetățenia britanică?",
            "ar": "كيف أتقدم بطلب للحصول على الجنسية البريطانية؟",
            "ur": "میں برطانوی شہریت کے لیے درخواست کیسے دوں؟"},
    "U04": {"en": "How do I register my baby's birth?",
            "pl": "Jak zarejestrować urodzenie mojego dziecka?",
            "ro": "Cum înregistrez nașterea copilului meu?",
            "ar": "كيف أسجل ولادة طفلي؟",
            "ur": "میں اپنے بچے کی پیدائش کا اندراج کیسے کرواؤں؟"},
    "U05": {"en": "What is the best treatment for back pain?",
            "pl": "Jakie jest najlepsze leczenie bólu pleców?",
            "ro": "Care este cel mai bun tratament pentru durerea de spate?",
            "ar": "ما هو أفضل علاج لآلام الظهر؟",
            "ur": "کمر درد کا بہترین علاج کیا ہے؟"},
    "U06": {"en": "How do I get a mortgage to buy my first home?",
            "pl": "Jak dostać kredyt hipoteczny na zakup pierwszego mieszkania?",
            "ro": "Cum obțin un credit ipotecar pentru a-mi cumpăra prima locuință?",
            "ar": "كيف أحصل على قرض عقاري لشراء أول منزل لي؟",
            "ur": "میں اپنا پہلا گھر خریدنے کے لیے مارگیج کیسے حاصل کروں؟"},
    "U07": {"en": "How do I register to vote?",
            "pl": "Jak zarejestrować się do głosowania?",
            "ro": "Cum mă înscriu pentru a vota?",
            "ar": "كيف أسجل للتصويت؟",
            "ur": "میں ووٹ ڈالنے کے لیے رجسٹریشن کیسے کرواؤں؟"},
    "U08": {"en": "Do I need a TV licence to watch TV online?",
            "pl": "Czy potrzebuję licencji telewizyjnej, żeby oglądać telewizję w internecie?",
            "ro": "Am nevoie de licență TV ca să mă uit la televizor online?",
            "ar": "هل أحتاج إلى ترخيص تلفزيون لمشاهدة التلفزيون عبر الإنترنت؟",
            "ur": "کیا مجھے آن لائن ٹی وی دیکھنے کے لیے ٹی وی لائسنس کی ضرورت ہے؟"},
    "U09": {"en": "How much does a driving test cost?",
            "pl": "Ile kosztuje egzamin na prawo jazdy?",
            "ro": "Cât costă examenul pentru permisul de conducere?",
            "ar": "كم تكلفة اختبار القيادة؟",
            "ur": "ڈرائیونگ ٹیسٹ کی فیس کتنی ہے؟"},
    "U10": {"en": "Can I get a free bus pass because I am over 60?",
            "pl": "Czy mogę dostać darmowy bilet autobusowy, bo mam ponad 60 lat?",
            "ro": "Pot primi un abonament gratuit de autobuz pentru că am peste 60 de ani?",
            "ar": "هل يمكنني الحصول على تذكرة حافلة مجانية لأن عمري أكثر من 60 عامًا؟",
            "ur": "کیا مجھے 60 سال سے زیادہ عمر ہونے کی وجہ سے مفت بس پاس مل سکتا ہے؟"},
}

if __name__ == "__main__":
    out = ROOT / "eval/unanswerable.csv"
    with open(out, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["qid", "lang", "question", "translation_status"])
        for qid, by_lang in QUESTIONS.items():
            for lang, q in by_lang.items():
                w.writerow([qid, lang, q, "source" if lang == "en" else "unverified"])
    print("wrote", out, 5 * len(QUESTIONS), "rows")


In [ ]:
%%writefile eval/evaluate_week6.py
"""Week 6 experiment: grounded answers, refusal rule and answer language.

Questions: 150 answerable (eval/queries.csv: 30 questions x 5 languages, each with a verbatim answer
phrase on its gold page) + 50 unanswerable (eval/unanswerable.csv: 10 topics not in the corpus).

Steps
  1. Search every question once with the week 5 pipeline (set-up F); cache to results/week6_search.json.
  2. Refusal rule 1: look at the reranker score of the top passage for answerable vs unanswerable
     questions and set the threshold just below the lowest score of any answerable question whose
     answer passage was retrieved, so no retrievable answer is refused at this step.
  3. Generate answers with Qwen3.5-4B in two modes (task T29):
       direct    - the LLM writes in the user's language
       translate - the LLM writes in English and NLLB translates it (non-English questions only)
     Answers are cached row by row in results/week6_answers.csv, so the run can be resumed.
     This first run ("v1") refused only on the threshold or on the model writing NO_ANSWER.
  3b. The v1 answers showed the model often explains in prose that the extracts do not answer the
     question instead of writing NO_ANSWER. Refusal rule 2 was therefore added: a separate yes/no call
     ("do the extracts answer the question?"), cached in results/week6_check.csv. Decoding is greedy and
     the answer prompt is unchanged, so an answer that passes the check is the same as in v1; the final
     result is v1 plus a refusal wherever the check says NO.
  3c. The first translate-mode run was invalid: although told to write in English, the model wrote in the
     question's language in 94-100% of answers, so NLLB was "translating" Polish, Romanian, Arabic or Urdu
     as if it were English, and citation markers split off on their own were dropped. Both were fixed
     (English instruction repeated after the question; stray markers kept) and translate mode was rerun.
     The invalid run is kept in results/week6_answers_v1.csv.
  4. Automatic checks per answer:
       refused        - the assistant refused (low relevance or the model said NO_ANSWER)
       cited          - at least one [n] citation, and every number refers to a given passage
       cites_answer   - a cited passage contains the question's verbatim answer phrase
       lang_ok        - the answer is in the question's language (lingua)
     These are checks of form and grounding, not of factual accuracy; accuracy is scored by hand with
     the week 7 rubric.
Run: python eval/evaluate_week6.py            (all 200 questions; about 2.5 hours on an Apple-silicon laptop)
     python eval/evaluate_week6.py --quick    (answers for 10 question topics x 5 languages only;
                                               the threshold is still calibrated on all 200)
Output: results/week6_by_mode.csv, results/week6_refusal.csv, results/week6_answers.csv
"""
import argparse
import json
import pathlib
import sys
import time

import pandas as pd

ROOT = pathlib.Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

import torch  # noqa: F401  (import before faiss on macOS, see retrieval.py)
from language import detect
from pipeline import SearchPipeline
from generate import citations, CITATION

SEARCH_CACHE = ROOT / "results/week6_search.json"
ANSWERS = ROOT / "results/week6_answers.csv"
CHECK = ROOT / "results/week6_check.csv"
QUICK = ["S01", "S05", "H03", "H06", "M01", "M04", "U01", "U03", "U04", "U07"]


def norm(s):
    return " ".join(str(s).replace("’", "'").lower().split())


def load_questions():
    a = pd.read_csv(ROOT / "eval/queries.csv")
    a["answerable"] = True
    u = pd.read_csv(ROOT / "eval/unanswerable.csv")
    u["answerable"], u["evidence"] = False, ""
    return pd.concat([a, u], ignore_index=True)


def run_search(qs, pipe):
    if SEARCH_CACHE.exists():
        return json.loads(SEARCH_CACHE.read_text())
    out = {}
    for r in qs.itertuples():
        out[f"{r.qid}-{r.lang}"] = pipe.search(r.question, k=3, lang=r.lang)
    SEARCH_CACHE.write_text(json.dumps(out, ensure_ascii=False, default=str))
    return out


def calibrate(qs, search):
    rows = []
    for r in qs.itertuples():
        s = search[f"{r.qid}-{r.lang}"]
        found = r.answerable and any(norm(r.evidence) in norm(p["text"]) for p in s["results"])
        rows.append({"qid": r.qid, "lang": r.lang, "answerable": r.answerable, "answer_retrieved": found,
                     "top_score": s["results"][0]["score"]})
    df = pd.DataFrame(rows)
    lowest = df[df.answer_retrieved].top_score.min()
    threshold = float(f"{lowest * 0.9:.3g}")  # 10% margin below the lowest retrievable answer
    return df, threshold


def strip_extras(text):
    return CITATION.sub("", text.split("\n\n")[0])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--quick", action="store_true", help="generate answers for a 10-topic subset only")
    quick = ap.parse_args().quick
    from assistant import Assistant
    from generate import Generator

    qs = load_questions()
    pipe = SearchPipeline()
    search = run_search(qs, pipe)
    scores, threshold = calibrate(qs, search)
    print("top reranker score, median by group:")
    print(scores.groupby("answerable").top_score.describe()[["min", "50%", "max"]].round(3).to_string())
    print(f"threshold = {threshold}")

    done = pd.read_csv(ANSWERS) if ANSWERS.exists() else pd.DataFrame()
    have = set(zip(done.qid, done.lang, done["mode"])) if len(done) else set()
    bot = Assistant(pipeline=pipe, generator=Generator(), threshold=threshold)
    rows = done.to_dict("records") if len(done) else []
    gen_qs = qs[qs.qid.isin(QUICK)] if quick else qs
    jobs = [(r, m) for r in gen_qs.itertuples() for m in ("direct", "translate")
            if not (m == "translate" and r.lang == "en") and (r.qid, r.lang, m) not in have]
    print(len(jobs), "answers to generate")
    for n, (r, mode) in enumerate(jobs, 1):
        t0 = time.time()
        s = search[f"{r.qid}-{r.lang}"]
        a = bot.answer_from(s, r.question, mode)
        cited = citations(a["answer"]) if not a["refused"] else []
        valid = bool(cited) and all(1 <= c <= len(s["results"]) for c in cited)
        cites_answer = r.answerable and any(norm(r.evidence) in norm(s["results"][c - 1]["text"])
                                            for c in cited if 1 <= c <= len(s["results"]))
        rows.append({"qid": r.qid, "lang": r.lang, "mode": mode, "answerable": r.answerable,
                     "question": r.question, "top_score": a["top_score"], "refused": a["refused"],
                     "reason": a["reason"], "cited": valid, "cites_answer": cites_answer,
                     "lang_ok": (not a["refused"]) and detect(strip_extras(a["answer"])) == r.lang,
                     "answer": a["answer"], "raw": a["raw"], "seconds": round(time.time() - t0, 1)})
        if n % 10 == 0 or n == len(jobs):
            pd.DataFrame(rows).to_csv(ANSWERS, index=False)
            print(f"{n}/{len(jobs)}", flush=True)

    df = pd.DataFrame(rows)
    df.to_csv(ANSWERS, index=False)

    # Refusal rule 2: yes/no answerability check (only for questions that pass the threshold).
    check = pd.read_csv(CHECK) if CHECK.exists() else pd.DataFrame(columns=["qid", "lang", "check_yes"])
    have = set(zip(check.qid, check.lang))
    new = []
    for r in gen_qs.itertuples():
        s = search[f"{r.qid}-{r.lang}"]
        if (r.qid, r.lang) not in have and s["results"][0]["score"] >= threshold:
            new.append({"qid": r.qid, "lang": r.lang, "check_yes": bot.generator.answerable(r.question, s["results"])})
    check = pd.concat([check, pd.DataFrame(new)], ignore_index=True)
    check.to_csv(CHECK, index=False)

    df = df.drop(columns=["check_yes"], errors="ignore").merge(check, on=["qid", "lang"], how="left")
    df.to_csv(ANSWERS, index=False)
    summarise(df, threshold)


def apply_check(df):
    """Final pipeline = v1 plus a refusal wherever the yes/no check said NO."""
    df = df.copy()
    blocked = (~df.refused) & (df.check_yes == False)  # noqa: E712  (NaN = not checked, below threshold)
    df.loc[blocked, ["refused", "cited", "cites_answer", "lang_ok"]] = [True, False, False, False]
    df.loc[blocked, "reason"] = "check: not in sources"
    return df


def summarise(df, threshold):
    out, refusal = [], []
    # v1 (no yes/no check) exists only for answers generated before the check was added: the original
    # direct-mode run. A fresh run (e.g. Colab) uses the check throughout, so only "final" is reported.
    pre_check = df[df["mode"] == "direct"]
    versions = [("final", apply_check(df), ("direct", "translate"))]
    if len(pre_check) and not (pre_check.reason == "check: not in sources").any():
        versions.insert(0, ("v1", df, ("direct",)))
    for version, d, modes in versions:
        by, ref = summarise_one(d[d["mode"].isin(modes)], modes)
        out.append(by.assign(version=version))
        refusal.append(ref.assign(version=version))
    by, ref = pd.concat(out), pd.concat(refusal)
    by.to_csv(ROOT / "results/week6_by_mode.csv", index=False)
    ref.to_csv(ROOT / "results/week6_refusal.csv", index=False)
    print(f"threshold {threshold}")
    print(by.to_string(index=False))
    print(ref.to_string(index=False))


def summarise_one(df, modes):
    ans, un = df[df.answerable], df[~df.answerable]
    rows = []
    for mode in modes:
        for lang in ["ALL", "en", "pl", "ro", "ar", "ur"]:
            if mode == "translate" and lang == "en":
                continue
            a = ans[(ans["mode"] == mode) & ((ans.lang == lang) | (lang == "ALL"))]
            u = un[(un["mode"] == mode) & ((un.lang == lang) | (lang == "ALL"))]
            if mode == "translate" and lang == "ALL":
                a, u = a[a.lang != "en"], u[u.lang != "en"]
            given = a[~a.refused]
            rows.append({"mode": mode, "lang": lang, "n_answerable": len(a),
                         "answered": round(1 - a.refused.mean(), 3),
                         "cited": round(given.cited.mean(), 3),
                         "cites_answer": round(given.cites_answer.mean(), 3),
                         "lang_ok": round(given.lang_ok.mean(), 3),
                         "n_unanswerable": len(u), "refused_unanswerable": round(u.refused.mean(), 3),
                         "sec_per_answer": round(df[(df["mode"] == mode) & ~df.refused].seconds.mean(), 1)})
    by = pd.DataFrame(rows)
    ref = df.groupby(["mode", "answerable", "reason"], dropna=False).size().rename("n").reset_index()
    return by, ref


if __name__ == "__main__":
    main()


In [ ]:
!{sys.executable} src/ingest.py

In [ ]:
!{sys.executable} src/chunk.py

In [ ]:
!{sys.executable} eval/build_queries.py

In [ ]:
!{sys.executable} eval/build_unanswerable.py

In [ ]:
!{sys.executable} -u eval/evaluate_week6.py --quick 2>&1 | grep -v -i -E 'warn|unauthenticated|Loading weights|falling back'

In [ ]:
import json, pathlib
import pandas as pd
ROOT = pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import torch
pd.set_option('display.max_colwidth', 120)

## 1. Choosing the LLM (T26)

**Qwen/Qwen3.5-4B** (Apache-2.0, not gated). It runs locally, so questions never leave the machine; the model card lists support for 201 languages and dialects; at 4B parameters (9.3 GB of weights) it fits a Colab T4 in float16. Thinking mode is switched off and decoding is greedy, so the same question always gives the same answer.

In [ ]:
import generate
print(generate.__doc__)

## 2. The grounded prompt (T27)

The three passages from the week 5 search stage are numbered [1] to [3]. The model may use only those facts, must end every sentence with a citation, and must write plain language in the requested language. Titles, links and fetch dates are never written by the model: they are looked up from the citation number.

In [ ]:
from generate import build_messages
search = json.loads((ROOT / 'results/week6_search.json').read_text())
example = search['M02-pl']
for m in build_messages('Czy potrzebuję dowodu adresu, żeby zarejestrować się u lekarza rodzinnego?', example['results'], 'pl'):
    print(f"--- {m['role']} ---\n{m['content'][:1500]}\n")

## 3. The assistant end to end

`src/assistant.py`: search (set-up F) > refusal rule 1 (reranker score below the threshold) > refusal rule 2 (yes/no check: do the passages answer the question?) > answer with citations > refusal rule 3 (model writes NO_ANSWER) > source cards > NHS 111/999 note when an NHS.uk page is cited. Loading the model takes a minute.

In [ ]:
from assistant import Assistant
bot = Assistant()

def ask(q):
    a = bot.ask(q)
    print(f"Q [{a['lang']}] {q}\n  top reranker score {a['top_score']:.3f}  refused={a['refused']} {a['reason'] or ''}\n")
    print(a['answer'])
    for s in a['sources']:
        print(f"  [{s['n']}] {s['title']} > {s['heading']}\n      {s['url']}  (fetched {s['fetched_at'][:10]})")
    print('\n' + '=' * 80 + '\n')

ask('Can I be fined for taking my child on holiday during term time?')
ask('Czy potrzebuję dowodu adresu, żeby zarejestrować się u lekarza rodzinnego?')
ask('How do I renew my UK passport?')   # not in the corpus: should be refused

## 4. Refusal rule 1: the relevance threshold (T28)

50 unanswerable questions (`eval/unanswerable.csv`: 10 topics not in the 38 pages x 5 languages) were added to the 150 answerable ones. The threshold is 0.9 x the lowest top reranker score of any answerable question whose answer passage was retrieved, so no retrievable answer is refused at this step.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
qs = pd.concat([pd.read_csv(ROOT / 'eval/queries.csv').assign(answerable=True),
                pd.read_csv(ROOT / 'eval/unanswerable.csv').assign(answerable=False)])
qs['top_score'] = [search[f'{r.qid}-{r.lang}']['results'][0]['score'] for r in qs.itertuples()]
from assistant import THRESHOLD
bins = np.logspace(-5, 0, 30)
plt.figure(figsize=(8, 3.2))
plt.hist(qs[qs.answerable].top_score, bins=bins, alpha=0.7, label='answerable (150)')
plt.hist(qs[~qs.answerable].top_score, bins=bins, alpha=0.7, label='unanswerable (50)')
plt.axvline(THRESHOLD, color='k', ls='--', label=f'threshold {THRESHOLD}')
plt.xscale('log'); plt.xlabel('top reranker score (log scale)'); plt.ylabel('questions'); plt.legend(frameon=False)
plt.tight_layout(); plt.show()
qs.groupby('answerable').top_score.describe()[['min', '50%', 'max']].round(3)

## 5. Refusal rule 2: why a separate yes/no check was added

In the first run (v1) the model was told to write NO_ANSWER when the passages do not answer the question. For 17 of the 18 unanswerable questions that passed the threshold it did not: it explained in a sentence that the extracts do not cover the topic. The fix is a separate, short yes/no call before the answer is written.

In [ ]:
ans = pd.read_csv(ROOT / 'results/week6_answers.csv')
soft = ans[(~ans.answerable) & (ans['mode'] == 'direct') & (ans.reason != 'low relevance')]
soft[['qid', 'lang', 'top_score', 'check_yes', 'answer']].head(8)

In [ ]:
ref = pd.read_csv(ROOT / 'results/week6_refusal.csv')
ref[ref['mode'] == 'direct'].pivot_table(index=['answerable', 'reason'], columns='version', values='n', dropna=False).fillna(0).astype(int)

## 6. Experiment: answer in the user's language - direct vs translate (T29)

* **direct**: the LLM writes in the user's language.
* **translate**: the LLM writes in English and NLLB translates each sentence (citations taken out and put back).

The first translate run was invalid: the model kept writing in the question's language (94-100% of drafts) despite being told to write English, so the instruction is now repeated after the question. That run is kept in `results/week6_answers_v1.csv`.

Automatic checks on each answer: **cited** = has valid [n] citations; **cites_answer** = a cited passage contains the question's verbatim answer phrase; **lang_ok** = written in the question's language; **refused_unanswerable** = share of the 50 unanswerable questions refused. These check form and grounding, not factual accuracy (scored by hand in week 7).

In [ ]:
by = pd.read_csv(ROOT / 'results/week6_by_mode.csv')
by[by.version == 'final'].drop(columns='version').set_index(['mode', 'lang'])

In [ ]:
langs = ['pl', 'ro', 'ar', 'ur']
t = by[(by.version == 'final') & by.lang.isin(langs)].pivot_table(index='lang', columns='mode', values='cites_answer').loc[langs]
ax = t.plot.bar(figsize=(7, 3.2), rot=0)
ax.set_ylim(0, 1); ax.set_ylabel('cites_answer'); ax.set_title('Answer cites the passage with the answer phrase')
ax.legend(frameon=False); plt.tight_layout(); plt.show()

### Same question, both modes

Translate mode: NLLB turned *GP surgery* into *chirurgii lekarskiej* (surgery as in an operation) and *eviction* into *wydalenia* (expulsion). (Author's reading of the Polish, not native-checked.)

In [ ]:
for qid in ['M02', 'H06']:
    for r in ans[(ans.qid == qid) & (ans.lang == 'pl')].itertuples():
        if r.mode == 'translate':
            print(f'{qid} English draft:\n  {r.raw[:400]}\n')
        print(f'{qid} pl {r.mode}:\n  {r.answer[:400]}\n')

## 7. What still goes wrong

In [ ]:
final_left = ans[(~ans.answerable) & (~ans.refused) & (ans.check_yes == True) & (ans['mode'] == 'direct')]
print(len(final_left), 'unanswerable questions still answered (direct mode)')
final_left[['qid', 'lang', 'question', 'answer']]

## Findings

* **No answerable question was refused** (0 of 150) by the threshold or the yes/no check.
* **Refusal of unanswerable questions: 33 of 50 in v1, 43 of 50 with the yes/no check** (direct mode). All 7 remaining cases are answers that say in prose that the sources do not cover it; none invents an answer, but they are not clean refusals. Four are 'register a birth' in pl/ro/ar/ur, where GP-registration passages look related.
* **Direct and translate are close** on the 120 non-English questions: 116 vs 112 cited, 101 vs 97 citing the answer passage - about one question per language. NLLB mistranslates key housing and health terms in otherwise correct answers.
* The first translate run was invalid (the model ignored 'write in English'); fixed and rerun.
* All answers were in the question's language in both modes (lingua).
* Decision: use **direct** mode with refusal rules 1-3 (simpler, no NLLB term errors); week 7 hand-scores a sample of both.

## Next (week 7)
Hand-score answer accuracy and faithfulness with a rubric on a gold set, compare against a no-retrieval baseline, and use held-out questions to check the refusal threshold.